In [1]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
from pathlib import Path
import math
import random
import numpy as np
import polars as pl
import torch
import torch.nn.functional as F
from torch import nn
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

PROJECT_PATH = Path("/content/drive/MyDrive/multimodal-fashion-recsys")
RAW_PATH = PROJECT_PATH / "data" / "raw"
PROCESSED_PATH = PROJECT_PATH / "data" / "processed"
CHECKPOINTS_PATH = PROJECT_PATH / "checkpoints"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

SEED = 1

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

print("Device:", DEVICE)

Device: cuda


In [3]:
train = pl.scan_parquet(PROCESSED_PATH / "train.parquet")
validation_ground_truth = pl.read_parquet(PROCESSED_PATH / "validation_ground_truth.parquet")
article_mapping = pl.read_parquet(PROCESSED_PATH / "article_mapping.parquet")
articles = pl.read_csv(RAW_PATH / "articles.csv")

NUM_ITEMS = article_mapping.height + 1

print("Items:", NUM_ITEMS - 1)

Items: 105542


In [4]:
METADATA_COLUMNS = [
    "product_code",
    "product_type_no",
    "graphical_appearance_no",
    "colour_group_code",
    "perceived_colour_value_id",
    "perceived_colour_master_id",
    "department_no",
    "index_group_no",
    "section_no",
    "garment_group_no"
]

In [5]:
article_metadata = (
    articles
    .select(["article_id"] + METADATA_COLUMNS)
    .join(article_mapping, on="article_id")
    .sort("article_idx")
)

article_metadata.head()

article_id,product_code,product_type_no,graphical_appearance_no,colour_group_code,perceived_colour_value_id,perceived_colour_master_id,department_no,index_group_no,section_no,garment_group_no,article_idx
i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,u32
108775015,108775,253,1010016,9,4,5,1676,1,16,1002,1
108775044,108775,253,1010016,10,3,9,1676,1,16,1002,2
108775051,108775,253,1010017,11,1,9,1676,1,16,1002,3
110065001,110065,306,1010016,9,4,5,1339,1,61,1017,4
110065002,110065,306,1010016,10,3,9,1339,1,61,1017,5


In [6]:
metadata = np.zeros(
    (NUM_ITEMS, len(METADATA_COLUMNS)),
    dtype=np.int64
)

metadata_sizes = []

article_indices = article_metadata["article_idx"].to_numpy()

for j, column in enumerate(METADATA_COLUMNS):
    values = article_metadata[column].to_numpy()
    _, encoded = np.unique(values, return_inverse=True)

    metadata[article_indices, j] = encoded + 1
    metadata_sizes.append(int(encoded.max()) + 2)

for column, size in zip(METADATA_COLUMNS, metadata_sizes):
    print(column, size - 1)

product_code 47224
product_type_no 132
graphical_appearance_no 30
colour_group_code 50
perceived_colour_value_id 8
perceived_colour_master_id 20
department_no 299
index_group_no 5
section_no 57
garment_group_no 21


In [7]:
print("Metadata shape:", metadata.shape)
print("Padding:", metadata[0])
print("First article:", metadata[1])

Metadata shape: (105543, 10)
Padding: [0 0 0 0 0 0 0 0 0 0]
First article: [ 1 49 17 10  5  6 48  1  9  2]


In [8]:
print("Articles without metadata:", np.all(metadata[1:] == 0, axis=1).sum())

Articles without metadata: 0


## Metadata-SASRec

In [9]:
checkpoint = torch.load(
    CHECKPOINTS_PATH / "sasrec_best.pt",
    map_location=DEVICE,
    weights_only=False
)

MAX_LEN = checkpoint["max_len"]
HIDDEN_DIM = checkpoint["hidden_dim"]
NUM_HEADS = checkpoint["num_heads"]
NUM_LAYERS = checkpoint["num_layers"]
DROPOUT = checkpoint["dropout"]

train_item_ids = train.select("article_idx").unique().collect()["article_idx"].to_numpy()

seen_item_mask = np.zeros(NUM_ITEMS, dtype=bool)
seen_item_mask[train_item_ids] = True

metadata_tensor = torch.from_numpy(metadata).long()

print("Base SASRec epoch:", checkpoint["epoch"])
print("Base SASRec MAP@12:", checkpoint["metrics"]["MAP@12"])
print("Train items:", seen_item_mask.sum())
print("Cold-start catalog items:", (~seen_item_mask[1:]).sum())

Base SASRec epoch: 8
Base SASRec MAP@12: 0.014934833159714998
Train items: 102967
Cold-start catalog items: 2575


In [10]:
def test_metadata_sasrec():
    test_model = MetadataSASRec(
        num_items=NUM_ITEMS,
        metadata_table=metadata_tensor,
        metadata_sizes=metadata_sizes,
        seen_item_mask=torch.from_numpy(seen_item_mask),
        max_len=MAX_LEN,
        hidden_dim=HIDDEN_DIM,
        num_heads=NUM_HEADS,
        num_layers=NUM_LAYERS,
        dropout=DROPOUT
    ).to(DEVICE)

    item_ids = torch.tensor([0, 1], device=DEVICE)
    embeddings = test_model.encode_items(item_ids)

    assert embeddings.shape == (2, HIDDEN_DIM)
    assert torch.allclose(embeddings[0], torch.zeros_like(embeddings[0]), atol=1e-6)

    print("Tests passed")

In [11]:
class MetadataSASRec(nn.Module):
    def __init__(
        self,
        num_items,
        metadata_table,
        metadata_sizes,
        seen_item_mask,
        max_len,
        hidden_dim,
        num_heads,
        num_layers,
        dropout,
        metadata_dim=32
    ):
        super().__init__()

        self.hidden_dim = hidden_dim

        self.item_embedding = nn.Embedding(num_items, hidden_dim, padding_idx=0)
        self.position_embedding = nn.Embedding(max_len, hidden_dim)

        self.metadata_embeddings = nn.ModuleList([
            nn.Embedding(size, metadata_dim, padding_idx=0)
            for size in metadata_sizes
        ])

        self.metadata_projection = nn.Sequential(
            nn.Linear(metadata_dim * len(metadata_sizes), hidden_dim),
            nn.GELU(),
            nn.LayerNorm(hidden_dim)
        )

        self.metadata_gate_logit = nn.Parameter(torch.tensor(-2.1972))

        self.register_buffer("metadata_table", metadata_table)
        self.register_buffer("seen_item_mask", seen_item_mask)

        self.dropout = nn.Dropout(dropout)

        layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim,
            nhead=num_heads,
            dim_feedforward=hidden_dim * 4,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True
        )

        self.transformer = nn.TransformerEncoder(
            layer,
            num_layers=num_layers,
            enable_nested_tensor=False
        )

        self.norm = nn.LayerNorm(hidden_dim)

        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

            if module.padding_idx is not None:
                with torch.no_grad():
                    module.weight[module.padding_idx].zero_()

        elif isinstance(module, nn.Linear):
            nn.init.xavier_uniform_(module.weight)

            if module.bias is not None:
                nn.init.zeros_(module.bias)

    def encode_items(self, item_ids):
        id_embeddings = self.item_embedding(item_ids)

        seen = self.seen_item_mask[item_ids].unsqueeze(-1)
        id_embeddings = id_embeddings * seen

        metadata_ids = self.metadata_table[item_ids]

        metadata_embeddings = [
            embedding(metadata_ids[..., j])
            for j, embedding in enumerate(self.metadata_embeddings)
        ]

        metadata_embeddings = torch.cat(metadata_embeddings, dim=-1)
        metadata_embeddings = self.metadata_projection(metadata_embeddings)

        metadata_weight = torch.sigmoid(self.metadata_gate_logit)

        embeddings = id_embeddings + metadata_weight * metadata_embeddings

        padding_mask = item_ids == 0
        return embeddings.masked_fill(padding_mask.unsqueeze(-1), 0.0)

    def forward(self, input_items):
        seq_len = input_items.size(1)
        positions = torch.arange(seq_len, device=input_items.device).unsqueeze(0)

        x = self.encode_items(input_items) * math.sqrt(self.hidden_dim)
        x = x + self.position_embedding(positions)
        x = self.dropout(x)

        padding_mask = input_items == 0
        x = x.masked_fill(padding_mask.unsqueeze(-1), 0.0)

        causal_mask = torch.triu(
            torch.ones(seq_len, seq_len, device=input_items.device, dtype=torch.bool),
            diagonal=1
        )

        x = self.transformer(x, mask=causal_mask, src_key_padding_mask=padding_mask)
        x = self.norm(x)

        return x.masked_fill(padding_mask.unsqueeze(-1), 0.0)

In [12]:
test_metadata_sasrec()

Tests passed


In [13]:
model = MetadataSASRec(
    num_items=NUM_ITEMS,
    metadata_table=metadata_tensor,
    metadata_sizes=metadata_sizes,
    seen_item_mask=torch.from_numpy(seen_item_mask),
    max_len=MAX_LEN,
    hidden_dim=HIDDEN_DIM,
    num_heads=NUM_HEADS,
    num_layers=NUM_LAYERS,
    dropout=DROPOUT
).to(DEVICE)

load_result = model.load_state_dict(
    checkpoint["model_state_dict"],
    strict=False
)

print("Missing keys:", load_result.missing_keys)
print("Unexpected keys:", load_result.unexpected_keys)
print("Metadata weight:", torch.sigmoid(model.metadata_gate_logit).item())

Missing keys: ['metadata_gate_logit', 'metadata_table', 'seen_item_mask', 'metadata_embeddings.0.weight', 'metadata_embeddings.1.weight', 'metadata_embeddings.2.weight', 'metadata_embeddings.3.weight', 'metadata_embeddings.4.weight', 'metadata_embeddings.5.weight', 'metadata_embeddings.6.weight', 'metadata_embeddings.7.weight', 'metadata_embeddings.8.weight', 'metadata_embeddings.9.weight', 'metadata_projection.0.weight', 'metadata_projection.0.bias', 'metadata_projection.2.weight', 'metadata_projection.2.bias']
Unexpected keys: []
Metadata weight: 0.10000219941139221


In [14]:
base_item_embeddings = checkpoint["model_state_dict"]["item_embedding.weight"]

print(torch.allclose(
    model.item_embedding.weight.cpu(),
    base_item_embeddings.cpu()
))

True


In [15]:
sequences = (
    train
    .sort(["customer_idx", "t_dat"])
    .group_by("customer_idx", maintain_order=True)
    .agg(pl.col("article_idx").alias("sequence"))
    .filter(pl.col("sequence").list.len() >= 2)
    .collect()
)

train_sequences = sequences["sequence"].to_list()

print("Training sequences:", len(train_sequences))

Training sequences: 1221097


In [16]:
class SASRecDataset(Dataset):
    def __init__(self, sequences, num_items, max_len):
        self.sequences = sequences
        self.num_items = num_items
        self.max_len = max_len

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        sequence = np.asarray(self.sequences[idx][-self.max_len - 1:], dtype=np.int64)

        inputs = sequence[:-1]
        positives = sequence[1:]

        input_items = np.zeros(self.max_len, dtype=np.int64)
        positive_items = np.zeros(self.max_len, dtype=np.int64)
        negative_items = np.zeros(self.max_len, dtype=np.int64)

        input_items[-len(inputs):] = inputs
        positive_items[-len(positives):] = positives

        negatives = np.random.randint(1, self.num_items, size=len(positives))
        invalid = np.isin(negatives, sequence)

        while invalid.any():
            negatives[invalid] = np.random.randint(1, self.num_items, size=invalid.sum())
            invalid = np.isin(negatives, sequence)

        negative_items[-len(negatives):] = negatives

        return (
            torch.from_numpy(input_items),
            torch.from_numpy(positive_items),
            torch.from_numpy(negative_items)
        )

In [17]:
dataset = SASRecDataset(train_sequences, NUM_ITEMS, MAX_LEN)

loader = DataLoader(
    dataset,
    batch_size=512,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True
)

In [18]:
val_users = validation_ground_truth.select("customer_idx")

val_history = (
    train
    .join(val_users.lazy(), on="customer_idx", how="semi")
    .sort(["customer_idx", "t_dat"])
    .group_by("customer_idx", maintain_order=True)
    .agg(pl.col("article_idx").tail(MAX_LEN).alias("history"))
    .collect()
)

history_map = dict(zip(
    val_history["customer_idx"].to_list(),
    val_history["history"].to_list()
))

val_user_ids = validation_ground_truth["customer_idx"].to_numpy()

val_sequences = np.zeros((len(val_user_ids), MAX_LEN), dtype=np.int64)
has_history = np.zeros(len(val_user_ids), dtype=bool)

for i, user_id in enumerate(val_user_ids):
    history = history_map.get(int(user_id), [])

    if history:
        history = history[-MAX_LEN:]
        val_sequences[i, -len(history):] = history
        has_history[i] = True

print("Validation users:", len(val_user_ids))
print("With history:", has_history.sum())
print("Without history:", (~has_history).sum())

Validation users: 72019
With history: 66624
Without history: 5395


In [19]:
all_item_ids_tensor = torch.arange(1, NUM_ITEMS, device=DEVICE)

print("Candidate items:", len(all_item_ids_tensor))

Candidate items: 105542


In [20]:
from datetime import date, timedelta

VAL_START = date(2020, 9, 9)
K = 12

recent_top12 = (
    train
    .filter(pl.col("t_dat") >= VAL_START - timedelta(days=14))
    .group_by("article_idx")
    .agg(pl.len().alias("count"))
    .sort("count", descending=True)
    .head(K)
    .collect()["article_idx"]
    .to_list()
)

In [21]:
def average_precision_at_k(actual, predicted, k=12):
    actual = set(actual)

    if not actual:
        return 0.0

    score = 0.0
    hits = 0
    seen = set()

    for i, item in enumerate(predicted[:k], 1):
        if item in actual and item not in seen:
            hits += 1
            score += hits / i

        seen.add(item)

    return score / min(len(actual), k)


def recall_at_k(actual, predicted, k=12):
    actual = set(actual)

    if not actual:
        return 0.0

    return len(actual.intersection(predicted[:k])) / len(actual)


def ndcg_at_k(actual, predicted, k=12):
    actual = set(actual)

    if not actual:
        return 0.0

    dcg = sum(1 / math.log2(i + 2) for i, item in enumerate(predicted[:k]) if item in actual)
    idcg = sum(1 / math.log2(i + 2) for i in range(min(len(actual), k)))

    return dcg / idcg

In [22]:
train_items_set = set(train_item_ids.tolist())
actuals = validation_ground_truth["actual"].to_list()

cold_actuals = [
    [item for item in actual if item not in train_items_set]
    for actual in actuals
]

cold_user_indices = [
    i for i, actual in enumerate(cold_actuals)
    if actual
]

cold_items = {
    item
    for actual in cold_actuals
    for item in actual
}

print("Validation users with cold-start purchases:", len(cold_user_indices))
print("Cold-start articles in validation:", len(cold_items))

Validation users with cold-start purchases: 8418
Cold-start articles in validation: 913


In [23]:
def evaluate_model(model):
    predictions = []
    model.eval()

    with torch.no_grad():
        item_embeddings = model.encode_items(all_item_ids_tensor)

        for start in tqdm(
            range(0, len(val_user_ids), 256),
            desc="Validation",
            leave=False
        ):
            end = min(start + 256, len(val_user_ids))

            batch_sequences = val_sequences[start:end]
            batch_history = has_history[start:end]

            batch_predictions = [recent_top12.copy() for _ in range(end - start)]

            valid_indices = np.flatnonzero(batch_history)

            if len(valid_indices) > 0:
                sequence_tensor = torch.from_numpy(
                    batch_sequences[valid_indices]
                ).to(DEVICE)

                with torch.amp.autocast("cuda", dtype=torch.float16):
                    hidden = model(sequence_tensor)
                    user_embeddings = hidden[:, -1]
                    scores = user_embeddings @ item_embeddings.T

                top_indices = scores.topk(K, dim=1).indices
                recommended = all_item_ids_tensor[top_indices].cpu().tolist()

                for position, recommendation in zip(valid_indices, recommended):
                    batch_predictions[position] = recommendation

            predictions.extend(batch_predictions)

    overall = {
        "MAP@12": sum(
            average_precision_at_k(a, p, K)
            for a, p in zip(actuals, predictions)
        ) / len(actuals),

        "Recall@12": sum(
            recall_at_k(a, p, K)
            for a, p in zip(actuals, predictions)
        ) / len(actuals),

        "NDCG@12": sum(
            ndcg_at_k(a, p, K)
            for a, p in zip(actuals, predictions)
        ) / len(actuals),

        "Coverage": len({
            item
            for prediction in predictions
            for item in prediction
        }) / (NUM_ITEMS - 1)
    }

    cold_predictions = [predictions[i] for i in cold_user_indices]
    cold_targets = [cold_actuals[i] for i in cold_user_indices]

    cold = {
        "Cold MAP@12": sum(
            average_precision_at_k(a, p, K)
            for a, p in zip(cold_targets, cold_predictions)
        ) / len(cold_targets),

        "Cold Recall@12": sum(
            recall_at_k(a, p, K)
            for a, p in zip(cold_targets, cold_predictions)
        ) / len(cold_targets)
    }

    return overall, cold

In [24]:
BASE_LR = 5e-5
METADATA_LR = 5e-4
WEIGHT_DECAY = 1e-5
EPOCHS = 4
PATIENCE = 2

metadata_params = []
base_params = []

for name, parameter in model.named_parameters():
    if name.startswith("metadata_"):
        metadata_params.append(parameter)
    else:
        base_params.append(parameter)

optimizer = torch.optim.AdamW([
    {
        "params": base_params,
        "lr": BASE_LR
    },
    {
        "params": metadata_params,
        "lr": METADATA_LR
    }
], weight_decay=WEIGHT_DECAY)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=EPOCHS,
    eta_min=1e-6
)

scaler = torch.amp.GradScaler("cuda")

In [25]:
best_map = -1.0
epochs_without_improvement = 0
training_history = []

CHECKPOINTS_PATH.mkdir(parents=True, exist_ok=True)

for epoch in range(1, EPOCHS + 1):
    model.train()

    total_loss = 0.0
    total_examples = 0

    pbar = tqdm(loader, desc=f"Epoch {epoch}/{EPOCHS}")

    for input_items, positive_items, negative_items in pbar:
        input_items = input_items.to(DEVICE, non_blocking=True)
        positive_items = positive_items.to(DEVICE, non_blocking=True)
        negative_items = negative_items.to(DEVICE, non_blocking=True)

        optimizer.zero_grad()

        with torch.amp.autocast("cuda", dtype=torch.float16):
            hidden = model(input_items)

            positive_embeddings = model.encode_items(positive_items)
            negative_embeddings = model.encode_items(negative_items)

            positive_scores = (hidden * positive_embeddings).sum(dim=-1)
            negative_scores = (hidden * negative_embeddings).sum(dim=-1)

            mask = positive_items != 0

            loss = -(
                F.logsigmoid(positive_scores[mask])
                + F.logsigmoid(-negative_scores[mask])
            ).mean()

        scaler.scale(loss).backward()

        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        scaler.step(optimizer)
        scaler.update()

        batch_size = input_items.size(0)
        total_loss += loss.item() * batch_size
        total_examples += batch_size

        pbar.set_postfix(loss=f"{total_loss / total_examples:.4f}")

    train_loss = total_loss / total_examples

    overall_metrics, cold_metrics = evaluate_model(model)

    training_history.append({
        "epoch": epoch,
        "loss": train_loss,
        **overall_metrics,
        **cold_metrics
    })

    print(
        f"Epoch {epoch}: "
        f"loss={train_loss:.4f}, "
        f"MAP@12={overall_metrics['MAP@12']:.6f}, "
        f"Cold MAP@12={cold_metrics['Cold MAP@12']:.6f}"
    )

    if overall_metrics["MAP@12"] > best_map:
        best_map = overall_metrics["MAP@12"]
        epochs_without_improvement = 0

        torch.save({
            "model_state_dict": model.state_dict(),
            "epoch": epoch,
            "overall_metrics": overall_metrics,
            "cold_metrics": cold_metrics,
            "metadata_columns": METADATA_COLUMNS,
            "metadata_sizes": metadata_sizes,
            "max_len": MAX_LEN,
            "hidden_dim": HIDDEN_DIM,
            "num_heads": NUM_HEADS,
            "num_layers": NUM_LAYERS,
            "dropout": DROPOUT
        }, CHECKPOINTS_PATH / "metadata_sasrec_best.pt")

        print("Saved best checkpoint")

    else:
        epochs_without_improvement += 1

    scheduler.step()

    if epochs_without_improvement >= PATIENCE:
        print("Early stopping")
        break

Epoch 1/4:   0%|          | 0/2385 [00:00<?, ?it/s]

Validation:   0%|          | 0/282 [00:00<?, ?it/s]

Epoch 1: loss=0.3617, MAP@12=0.015434, Cold MAP@12=0.000000
Saved best checkpoint


Epoch 2/4:   0%|          | 0/2385 [00:00<?, ?it/s]

Validation:   0%|          | 0/282 [00:00<?, ?it/s]

...

Validation:   0%|          | 0/282 [00:00<?, ?it/s]

Epoch 3: loss=0.3337, MAP@12=0.015901, Cold MAP@12=0.000000
Saved best checkpoint


Epoch 4/4:   0%|          | 0/2385 [00:00<?, ?it/s]

Validation:   0%|          | 0/282 [00:00<?, ?it/s]

Epoch 4: loss=0.3312, MAP@12=0.016332, Cold MAP@12=0.000000
Saved best checkpoint


In [26]:
training_history = pl.DataFrame(training_history)

training_history

epoch,loss,MAP@12,Recall@12,NDCG@12,Coverage,Cold MAP@12,Cold Recall@12
i64,f64,f64,f64,f64,f64,f64,f64
1,0.361664,0.015434,0.038086,0.024891,0.17972,0.0,0.0
2,0.33691,0.015788,0.038847,0.025394,0.182354,0.0,0.0
3,0.333656,0.015901,0.039009,0.025516,0.189489,0.0,0.0
4,0.331191,0.016332,0.039601,0.02606,0.192786,0.0,0.0


## Cold-start Diagnostics

In [27]:
best_checkpoint = torch.load(
    CHECKPOINTS_PATH / "metadata_sasrec_best.pt",
    map_location=DEVICE,
    weights_only=False
)

print("Best epoch:", best_checkpoint["epoch"])
print("Overall:", best_checkpoint["overall_metrics"])
print("Cold-start:", best_checkpoint["cold_metrics"])
print("Metadata weight:", torch.sigmoid(model.metadata_gate_logit).item())

Best epoch: 4
Overall: {'MAP@12': 0.01633216557929502, 'Recall@12': 0.039601327949883566, 'NDCG@12': 0.026060213586694306, 'Coverage': 0.19278581038828144}
Cold-start: {'Cold MAP@12': 0.0, 'Cold Recall@12': 0.0}
Metadata weight: 0.07990002632141113


In [28]:
train_item_set = set(train_item_ids.tolist())

train_metadata = article_metadata.filter(
    pl.col("article_idx").is_in(train_item_ids.tolist())
)

cold_metadata = article_metadata.filter(
    ~pl.col("article_idx").is_in(train_item_ids.tolist())
)

for column in METADATA_COLUMNS:
    train_values = set(train_metadata[column].to_list())
    cold_values = cold_metadata[column].to_list()

    known = sum(value in train_values for value in cold_values)

    print(
        column,
        f"{known}/{len(cold_values)} known",
        f"({known / len(cold_values):.1%})"
    )

product_code 1180/2575 known (45.8%)
product_type_no 2567/2575 known (99.7%)
graphical_appearance_no 2575/2575 known (100.0%)
colour_group_code 2575/2575 known (100.0%)
perceived_colour_value_id 2575/2575 known (100.0%)
perceived_colour_master_id 2575/2575 known (100.0%)
department_no 2575/2575 known (100.0%)
index_group_no 2575/2575 known (100.0%)
section_no 2575/2575 known (100.0%)
garment_group_no 2575/2575 known (100.0%)


In [29]:
def cold_recall_at_k(actual, predicted, k):
    if not actual:
        return 0.0

    actual = set(actual)
    return len(actual.intersection(predicted[:k])) / min(len(actual), k)

In [30]:
def evaluate_cold_top100(model):
    predictions = []
    model.eval()

    with torch.no_grad():
        item_embeddings = model.encode_items(all_item_ids_tensor)

        for start in tqdm(
            range(0, len(val_user_ids), 256),
            desc="Cold-start Top-100"
        ):
            end = min(start + 256, len(val_user_ids))

            batch_sequences = val_sequences[start:end]
            batch_history = has_history[start:end]

            batch_predictions = [[] for _ in range(end - start)]

            valid_indices = np.flatnonzero(batch_history)

            if len(valid_indices) > 0:
                sequence_tensor = torch.from_numpy(
                    batch_sequences[valid_indices]
                ).to(DEVICE)

                with torch.amp.autocast("cuda", dtype=torch.float16):
                    hidden = model(sequence_tensor)
                    user_embeddings = hidden[:, -1]
                    scores = user_embeddings @ item_embeddings.T

                top_indices = scores.topk(100, dim=1).indices
                recommended = all_item_ids_tensor[top_indices].cpu().tolist()

                for position, recommendation in zip(valid_indices, recommended):
                    batch_predictions[position] = recommendation

            predictions.extend(batch_predictions)

    cold_predictions = [predictions[i] for i in cold_user_indices]
    cold_targets = [cold_actuals[i] for i in cold_user_indices]

    return sum(
        cold_recall_at_k(a, p, 100)
        for a, p in zip(cold_targets, cold_predictions)
    ) / len(cold_targets)

In [31]:
cold_recall_100 = evaluate_cold_top100(model)

print("Cold Recall@100:", cold_recall_100)

Cold-start Top-100:   0%|          | 0/282 [00:00<?, ?it/s]

Cold Recall@100: 0.0


In [32]:
with torch.no_grad():
    all_ids = torch.arange(1, NUM_ITEMS, device=DEVICE)
    all_embeddings = model.encode_items(all_ids)

    norms = all_embeddings.norm(dim=1).cpu().numpy()

seen_norms = norms[seen_item_mask[1:]]
cold_norms = norms[~seen_item_mask[1:]]

print("Seen mean norm:", seen_norms.mean())
print("Seen median norm:", np.median(seen_norms))
print("Cold mean norm:", cold_norms.mean())
print("Cold median norm:", np.median(cold_norms))

Seen mean norm: 0.9272566
Seen median norm: 0.9311689
Cold mean norm: 0.80230886
Cold median norm: 0.93343127


In [33]:
with torch.no_grad():
    item_ids = torch.arange(1, NUM_ITEMS, device=DEVICE)

    id_embeddings = model.item_embedding(item_ids)

    metadata_ids = model.metadata_table[item_ids]

    metadata_embeddings = [
        embedding(metadata_ids[..., j])
        for j, embedding in enumerate(model.metadata_embeddings)
    ]

    metadata_embeddings = torch.cat(metadata_embeddings, dim=-1)
    metadata_embeddings = model.metadata_projection(metadata_embeddings)

    id_norms = id_embeddings.norm(dim=1).cpu().numpy()
    metadata_norms = metadata_embeddings.norm(dim=1).cpu().numpy()

print("ID mean norm:", id_norms.mean())
print("Metadata mean norm:", metadata_norms.mean())

ID mean norm: 0.8650374
Metadata mean norm: 2.4310346


## Metadata-SASRec v2

In [52]:
METADATA_COLUMNS_V2 = [
    "product_type_no",
    "graphical_appearance_no",
    "colour_group_code",
    "perceived_colour_value_id",
    "perceived_colour_master_id",
    "department_no",
    "index_group_no",
    "section_no",
    "garment_group_no"
]

article_metadata_v2 = (
    articles
    .select(["article_id"] + METADATA_COLUMNS_V2)
    .join(article_mapping, on="article_id")
    .sort("article_idx")
)

metadata_v2 = np.zeros(
    (NUM_ITEMS, len(METADATA_COLUMNS_V2)),
    dtype=np.int64
)

metadata_sizes_v2 = []
article_indices_v2 = article_metadata_v2["article_idx"].to_numpy()

for j, column in enumerate(METADATA_COLUMNS_V2):
    train_values = (
        article_metadata_v2
        .filter(pl.col("article_idx").is_in(train_item_ids.tolist()))
        [column]
        .unique()
        .to_list()
    )

    value_to_idx = {
        value: idx + 1
        for idx, value in enumerate(train_values)
    }

    encoded = np.array([
        value_to_idx.get(value, 0)
        for value in article_metadata_v2[column].to_list()
    ])

    metadata_v2[article_indices_v2, j] = encoded
    metadata_sizes_v2.append(len(value_to_idx) + 1)

train_item_ids_np = train_item_ids.astype(np.int64)

cold_item_ids = np.where(~seen_item_mask)[0]
cold_item_ids = cold_item_ids[cold_item_ids != 0]

for j, column in enumerate(METADATA_COLUMNS_V2):
    unknown = (metadata_v2[cold_item_ids, j] == 0).mean()
    print(column, f"unknown: {unknown:.1%}")

base_item_weight = checkpoint["model_state_dict"]["item_embedding.weight"]

train_ids_tensor = torch.from_numpy(train_item_ids_np).long().to(
    base_item_weight.device
)

ITEM_NORM_TARGET = (
    base_item_weight[train_ids_tensor]
    .norm(dim=1)
    .mean()
    .item()
)

print("Target item norm:", ITEM_NORM_TARGET)

product_type_no unknown: 0.3%
graphical_appearance_no unknown: 0.0%
colour_group_code unknown: 0.0%
perceived_colour_value_id unknown: 0.0%
perceived_colour_master_id unknown: 0.0%
department_no unknown: 0.0%
index_group_no unknown: 0.0%
section_no unknown: 0.0%
garment_group_no unknown: 0.0%
Target item norm: 0.8444344997406006


In [53]:
class SASRecDatasetV2(Dataset):
    def __init__(self, sequences, train_item_ids, max_len):
        self.sequences = sequences
        self.train_item_ids = train_item_ids
        self.max_len = max_len

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        sequence = np.asarray(
            self.sequences[idx][-self.max_len - 1:],
            dtype=np.int64
        )

        inputs = sequence[:-1]
        positives = sequence[1:]

        input_items = np.zeros(self.max_len, dtype=np.int64)
        positive_items = np.zeros(self.max_len, dtype=np.int64)
        negative_items = np.zeros(self.max_len, dtype=np.int64)

        input_items[-len(inputs):] = inputs
        positive_items[-len(positives):] = positives

        negatives = np.random.choice(
            self.train_item_ids,
            size=len(positives),
            replace=True
        )

        invalid = np.isin(negatives, sequence)

        while invalid.any():
            negatives[invalid] = np.random.choice(
                self.train_item_ids,
                size=invalid.sum(),
                replace=True
            )

            invalid = np.isin(negatives, sequence)

        negative_items[-len(negatives):] = negatives

        return (
            torch.from_numpy(input_items),
            torch.from_numpy(positive_items),
            torch.from_numpy(negative_items)
        )

In [54]:
class MetadataSASRecV2(nn.Module):
    def __init__(
        self,
        num_items,
        metadata_table,
        metadata_sizes,
        seen_item_mask,
        max_len,
        hidden_dim,
        num_heads,
        num_layers,
        dropout,
        item_norm_target,
        metadata_dim=32
    ):
        super().__init__()

        self.hidden_dim = hidden_dim

        self.register_buffer(
            "item_norm_target",
            torch.tensor(float(item_norm_target))
        )

        self.item_embedding = nn.Embedding(
            num_items,
            hidden_dim,
            padding_idx=0
        )

        self.position_embedding = nn.Embedding(
            max_len,
            hidden_dim
        )

        self.metadata_embeddings = nn.ModuleList([
            nn.Embedding(
                size,
                metadata_dim,
                padding_idx=0
            )
            for size in metadata_sizes
        ])

        self.metadata_projection = nn.Sequential(
            nn.Linear(
                metadata_dim * len(metadata_sizes),
                hidden_dim
            ),
            nn.GELU(),
            nn.LayerNorm(hidden_dim)
        )

        self.metadata_gate_logit = nn.Parameter(
            torch.tensor(-2.9444)
        )

        self.register_buffer(
            "metadata_table",
            metadata_table
        )

        self.register_buffer(
            "seen_item_mask",
            seen_item_mask
        )

        self.dropout = nn.Dropout(dropout)

        layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim,
            nhead=num_heads,
            dim_feedforward=hidden_dim * 4,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True
        )

        self.transformer = nn.TransformerEncoder(
            layer,
            num_layers=num_layers,
            enable_nested_tensor=False
        )

        self.norm = nn.LayerNorm(hidden_dim)

    def encode_items(self, item_ids):
        id_embeddings = self.item_embedding(item_ids)

        metadata_ids = self.metadata_table[item_ids]

        metadata_embeddings = [
            embedding(metadata_ids[..., j])
            for j, embedding in enumerate(self.metadata_embeddings)
        ]

        metadata_embeddings = torch.cat(
            metadata_embeddings,
            dim=-1
        )

        metadata_embeddings = self.metadata_projection(
            metadata_embeddings
        )

        metadata_embeddings = F.normalize(
            metadata_embeddings,
            p=2,
            dim=-1
        )

        metadata_embeddings = (
            metadata_embeddings
            * self.item_norm_target
        )

        metadata_weight = torch.sigmoid(
            self.metadata_gate_logit
        )

        seen = self.seen_item_mask[
            item_ids
        ].unsqueeze(-1)

        seen_embeddings = (
            id_embeddings
            + metadata_weight * metadata_embeddings
        )

        cold_embeddings = metadata_embeddings

        embeddings = torch.where(
            seen,
            seen_embeddings,
            cold_embeddings
        )

        padding_mask = item_ids == 0

        return embeddings.masked_fill(
            padding_mask.unsqueeze(-1),
            0.0
        )

    def forward(self, input_items):
        seq_len = input_items.size(1)

        positions = torch.arange(
            seq_len,
            device=input_items.device
        ).unsqueeze(0)

        x = self.encode_items(input_items) * math.sqrt(self.hidden_dim)
        x = x + self.position_embedding(positions)
        x = self.dropout(x)

        padding_mask = input_items == 0
        x = x.masked_fill(
            padding_mask.unsqueeze(-1),
            0.0
        )

        causal_mask = torch.triu(
            torch.ones(
                seq_len,
                seq_len,
                device=input_items.device,
                dtype=torch.bool
            ),
            diagonal=1
        )

        x = self.transformer(
            x,
            mask=causal_mask,
            src_key_padding_mask=padding_mask
        )

        x = self.norm(x)

        return x.masked_fill(
            padding_mask.unsqueeze(-1),
            0.0
        )

In [55]:
metadata_tensor_v2 = torch.from_numpy(metadata_v2).long()

model_v2 = MetadataSASRecV2(
    num_items=NUM_ITEMS,
    metadata_table=metadata_tensor_v2,
    metadata_sizes=metadata_sizes_v2,
    seen_item_mask=torch.from_numpy(seen_item_mask),
    max_len=MAX_LEN,
    hidden_dim=HIDDEN_DIM,
    num_heads=NUM_HEADS,
    num_layers=NUM_LAYERS,
    dropout=DROPOUT,
    item_norm_target=ITEM_NORM_TARGET
).to(DEVICE)

base_state = checkpoint["model_state_dict"]
model_state = model_v2.state_dict()

compatible_state = {
    key: value
    for key, value in base_state.items()
    if key in model_state
    and model_state[key].shape == value.shape
}

load_result = model_v2.load_state_dict(
    compatible_state,
    strict=False
)

print("Loaded parameters:", len(compatible_state))
print("Unexpected keys:", load_result.unexpected_keys)
print(
    "Metadata weight:",
    torch.sigmoid(model_v2.metadata_gate_logit).item()
)

print(
    "Base item embeddings loaded:",
    torch.allclose(
        model_v2.item_embedding.weight,
        base_state["item_embedding.weight"]
    )
)

Loaded parameters: 40
Unexpected keys: []
Metadata weight: 0.0500018484890461
Base item embeddings loaded: True


In [56]:
dataset_v2 = SASRecDatasetV2(
    train_sequences,
    train_item_ids_np,
    MAX_LEN
)

loader_v2 = DataLoader(
    dataset_v2,
    batch_size=512,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True
)

all_item_ids_tensor = torch.arange(
    1,
    NUM_ITEMS,
    device=DEVICE
)

with torch.no_grad():
    embeddings_v2 = model_v2.encode_items(
        all_item_ids_tensor
    )

    norms_v2 = (
        embeddings_v2
        .norm(dim=1)
        .cpu()
        .numpy()
    )

seen_norms_v2 = norms_v2[
    seen_item_mask[1:]
]

cold_norms_v2 = norms_v2[
    ~seen_item_mask[1:]
]

print("Seen mean norm:", seen_norms_v2.mean())
print("Cold mean norm:", cold_norms_v2.mean())
print("Candidate items:", len(all_item_ids_tensor))

Seen mean norm: 0.8451841
Cold mean norm: 0.84443456
Candidate items: 105542


In [57]:
def recall_at_k(actual, predicted, k=12):
    actual = set(actual)

    if not actual:
        return 0.0

    return (
        len(actual.intersection(predicted[:k]))
        / min(len(actual), k)
    )


def evaluate_model_v2(model):
    predictions = []
    predictions_top100 = []

    model.eval()

    with torch.no_grad():
        item_embeddings = model.encode_items(
            all_item_ids_tensor
        )

        for start in tqdm(
            range(0, len(val_user_ids), 256),
            desc="Validation",
            leave=True
        ):
            end = min(
                start + 256,
                len(val_user_ids)
            )

            batch_sequences = val_sequences[start:end]
            batch_history = has_history[start:end]

            batch_predictions = [
                recent_top12.copy()
                for _ in range(end - start)
            ]

            batch_top100 = [
                []
                for _ in range(end - start)
            ]

            valid_indices = np.flatnonzero(
                batch_history
            )

            if len(valid_indices) > 0:
                sequence_tensor = torch.from_numpy(
                    batch_sequences[valid_indices]
                ).to(DEVICE)

                with torch.amp.autocast(
                    "cuda",
                    dtype=torch.float16
                ):
                    hidden = model(
                        sequence_tensor
                    )

                    user_embeddings = hidden[:, -1]

                    scores = (
                        user_embeddings
                        @ item_embeddings.T
                    )

                top_indices = scores.topk(
                    100,
                    dim=1
                ).indices

                recommended = all_item_ids_tensor[
                    top_indices
                ].cpu().tolist()

                for position, recommendation in zip(
                    valid_indices,
                    recommended
                ):
                    batch_top100[position] = recommendation
                    batch_predictions[position] = recommendation[:K]

            predictions.extend(
                batch_predictions
            )

            predictions_top100.extend(
                batch_top100
            )

    overall = {
        "MAP@12": sum(
            average_precision_at_k(a, p, K)
            for a, p in zip(actuals, predictions)
        ) / len(actuals),

        "Recall@12": sum(
            recall_at_k(a, p, K)
            for a, p in zip(actuals, predictions)
        ) / len(actuals),

        "NDCG@12": sum(
            ndcg_at_k(a, p, K)
            for a, p in zip(actuals, predictions)
        ) / len(actuals),

        "Coverage": len({
            item
            for prediction in predictions
            for item in prediction
        }) / (NUM_ITEMS - 1)
    }

    cold_predictions_12 = [
        predictions[i]
        for i in cold_user_indices
    ]

    cold_predictions_100 = [
        predictions_top100[i]
        for i in cold_user_indices
    ]

    cold_targets = [
        cold_actuals[i]
        for i in cold_user_indices
    ]

    cold = {
        "Cold MAP@12": sum(
            average_precision_at_k(a, p, K)
            for a, p in zip(
                cold_targets,
                cold_predictions_12
            )
        ) / len(cold_targets),

        "Cold Recall@12": sum(
            recall_at_k(a, p, K)
            for a, p in zip(
                cold_targets,
                cold_predictions_12
            )
        ) / len(cold_targets),

        "Cold Recall@100": sum(
            recall_at_k(a, p, 100)
            for a, p in zip(
                cold_targets,
                cold_predictions_100
            )
        ) / len(cold_targets)
    }

    return overall, cold

In [58]:
overall_before, cold_before = evaluate_model_v2(
    model_v2
)

print("Before fine-tuning:")
print("Overall:", overall_before)
print("Cold:", cold_before)

Validation:   0%|          | 0/282 [00:00<?, ?it/s]

Before fine-tuning:
Overall: {'MAP@12': 0.014584065252031982, 'Recall@12': 0.03639684819824783, 'NDCG@12': 0.02365589378761813, 'Coverage': 0.16896590930624775}
Cold: {'Cold MAP@12': 2.121304687234837e-06, 'Cold Recall@12': 1.6970437497878696e-05, 'Cold Recall@100': 0.002548394030931451}


In [59]:
def encode_metadata_only(model, item_ids):
    metadata_ids = model.metadata_table[item_ids]

    metadata_embeddings = [
        embedding(metadata_ids[..., j])
        for j, embedding in enumerate(model.metadata_embeddings)
    ]

    metadata_embeddings = torch.cat(
        metadata_embeddings,
        dim=-1
    )

    metadata_embeddings = model.metadata_projection(
        metadata_embeddings
    )

    metadata_embeddings = F.normalize(
        metadata_embeddings,
        p=2,
        dim=-1
    )

    metadata_embeddings = (
        metadata_embeddings
        * model.item_norm_target
    )

    padding_mask = item_ids == 0

    return metadata_embeddings.masked_fill(
        padding_mask.unsqueeze(-1),
        0.0
    )

In [60]:
BASE_LR = 1e-5
METADATA_LR = 3e-4
WEIGHT_DECAY = 1e-5

EPOCHS = 6
PATIENCE = 2
COLD_LOSS_WEIGHT = 0.5

metadata_params = []
base_params = []

for name, parameter in model_v2.named_parameters():
    if name.startswith("metadata_"):
        metadata_params.append(parameter)
    else:
        base_params.append(parameter)

optimizer_v2 = torch.optim.AdamW([
    {
        "params": base_params,
        "lr": BASE_LR
    },
    {
        "params": metadata_params,
        "lr": METADATA_LR
    }
], weight_decay=WEIGHT_DECAY)

scheduler_v2 = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer_v2,
    T_max=EPOCHS,
    eta_min=1e-6
)

scaler_v2 = torch.amp.GradScaler("cuda")

In [61]:
best_map = overall_before["MAP@12"]
best_cold_recall = cold_before["Cold Recall@100"]

epochs_without_improvement = 0
training_history_v2 = []

CHECKPOINTS_PATH.mkdir(
    parents=True,
    exist_ok=True
)

for epoch in range(1, EPOCHS + 1):
    model_v2.train()

    total_loss = 0.0
    total_main_loss = 0.0
    total_cold_loss = 0.0
    total_examples = 0

    pbar = tqdm(
        loader_v2,
        desc=f"Epoch {epoch}/{EPOCHS}"
    )

    for input_items, positive_items, negative_items in pbar:
        input_items = input_items.to(
            DEVICE,
            non_blocking=True
        )

        positive_items = positive_items.to(
            DEVICE,
            non_blocking=True
        )

        negative_items = negative_items.to(
            DEVICE,
            non_blocking=True
        )

        optimizer_v2.zero_grad()

        with torch.amp.autocast(
            "cuda",
            dtype=torch.float16
        ):
            hidden = model_v2(input_items)

            positive_embeddings = model_v2.encode_items(
                positive_items
            )

            negative_embeddings = model_v2.encode_items(
                negative_items
            )

            positive_scores = (
                hidden * positive_embeddings
            ).sum(dim=-1)

            negative_scores = (
                hidden * negative_embeddings
            ).sum(dim=-1)

            mask = positive_items != 0

            main_loss = -(
                F.logsigmoid(
                    positive_scores[mask]
                )
                + F.logsigmoid(
                    -negative_scores[mask]
                )
            ).mean()

            positive_metadata = encode_metadata_only(
                model_v2,
                positive_items
            )

            negative_metadata = encode_metadata_only(
                model_v2,
                negative_items
            )

            cold_positive_scores = (
                hidden * positive_metadata
            ).sum(dim=-1)

            cold_negative_scores = (
                hidden * negative_metadata
            ).sum(dim=-1)

            cold_loss = -(
                F.logsigmoid(
                    cold_positive_scores[mask]
                )
                + F.logsigmoid(
                    -cold_negative_scores[mask]
                )
            ).mean()

            loss = (
                main_loss
                + COLD_LOSS_WEIGHT * cold_loss
            )

        scaler_v2.scale(loss).backward()

        scaler_v2.unscale_(
            optimizer_v2
        )

        torch.nn.utils.clip_grad_norm_(
            model_v2.parameters(),
            1.0
        )

        scaler_v2.step(
            optimizer_v2
        )

        scaler_v2.update()

        batch_size = input_items.size(0)

        total_loss += (
            loss.item() * batch_size
        )

        total_main_loss += (
            main_loss.item() * batch_size
        )

        total_cold_loss += (
            cold_loss.item() * batch_size
        )

        total_examples += batch_size

        pbar.set_postfix(
            loss=f"{total_loss / total_examples:.4f}",
            main=f"{total_main_loss / total_examples:.4f}",
            cold=f"{total_cold_loss / total_examples:.4f}"
        )

    train_loss = (
        total_loss / total_examples
    )

    overall_metrics, cold_metrics = evaluate_model_v2(
        model_v2
    )

    metadata_weight = torch.sigmoid(
        model_v2.metadata_gate_logit
    ).item()

    training_history_v2.append({
        "epoch": epoch,
        "loss": train_loss,
        **overall_metrics,
        **cold_metrics,
        "metadata_weight": metadata_weight
    })

    print(
        f"Epoch {epoch}: "
        f"loss={train_loss:.4f}, "
        f"MAP@12={overall_metrics['MAP@12']:.6f}, "
        f"Cold MAP@12={cold_metrics['Cold MAP@12']:.6f}, "
        f"Cold Recall@100={cold_metrics['Cold Recall@100']:.6f}, "
        f"metadata_weight={metadata_weight:.4f}"
    )

    if overall_metrics["MAP@12"] > best_map:
        best_map = overall_metrics["MAP@12"]
        best_cold_recall = cold_metrics[
            "Cold Recall@100"
        ]

        epochs_without_improvement = 0

        torch.save({
            "model_state_dict": model_v2.state_dict(),
            "epoch": epoch,
            "overall_metrics": overall_metrics,
            "cold_metrics": cold_metrics,
            "metadata_columns": METADATA_COLUMNS_V2,
            "metadata_sizes": metadata_sizes_v2,
            "item_norm_target": ITEM_NORM_TARGET,
            "max_len": MAX_LEN,
            "hidden_dim": HIDDEN_DIM,
            "num_heads": NUM_HEADS,
            "num_layers": NUM_LAYERS,
            "dropout": DROPOUT
        }, CHECKPOINTS_PATH / "metadata_sasrec_v2_best.pt")

        print("Saved best checkpoint")

    else:
        epochs_without_improvement += 1

    scheduler_v2.step()

    if epochs_without_improvement >= PATIENCE:
        print("Early stopping")
        break

Epoch 1/6:   0%|          | 0/2385 [00:00<?, ?it/s]

Validation:   0%|          | 0/282 [00:00<?, ?it/s]

Epoch 1: loss=0.7397, MAP@12=0.014997, Cold MAP@12=0.000000, Cold Recall@100=0.000891, metadata_weight=0.0734
Saved best checkpoint


Epoch 2/6:   0%|          | 0/2385 [00:00<?, ?it/s]

Validation:   0%|          | 0/282 [00:00<?, ?it/s]

...

Validation:   0%|          | 0/282 [00:00<?, ?it/s]

Epoch 5: loss=0.7017, MAP@12=0.015092, Cold MAP@12=0.000000, Cold Recall@100=0.001426, metadata_weight=0.1211
Saved best checkpoint


Epoch 6/6:   0%|          | 0/2385 [00:00<?, ?it/s]

Validation:   0%|          | 0/282 [00:00<?, ?it/s]

Epoch 6: loss=0.7006, MAP@12=0.015156, Cold MAP@12=0.000000, Cold Recall@100=0.001129, metadata_weight=0.1218
Saved best checkpoint


In [62]:
training_history_v2 = pl.DataFrame(
    training_history_v2
)

training_history_v2

epoch,loss,MAP@12,Recall@12,NDCG@12,Coverage,Cold MAP@12,Cold Recall@12,Cold Recall@100,metadata_weight
i64,f64,f64,f64,f64,f64,f64,f64,f64,f64
1,0.739699,0.014997,0.037809,0.024427,0.16475,0.0,0.0,0.000891,0.073377
2,0.714121,0.015016,0.037827,0.024436,0.166019,0.0,0.0,0.000772,0.100002
3,0.707698,0.01498,0.037769,0.024392,0.166474,0.0,0.0,0.000653,0.11295
4,0.703935,0.015043,0.037948,0.024482,0.167204,0.0,0.0,0.00101,0.118573
5,0.701743,0.015092,0.03796,0.024529,0.166853,0.0,0.0,0.001426,0.121118
6,0.700621,0.015156,0.037973,0.024587,0.167478,0.0,0.0,0.001129,0.121811


In [63]:
best_v2_checkpoint = torch.load(
    CHECKPOINTS_PATH / "metadata_sasrec_v2_best.pt",
    map_location=DEVICE,
    weights_only=False
)

print("Best epoch:", best_v2_checkpoint["epoch"])
print("Overall:", best_v2_checkpoint["overall_metrics"])
print("Cold:", best_v2_checkpoint["cold_metrics"])

Best epoch: 6
Overall: {'MAP@12': 0.01515559086099657, 'Recall@12': 0.037973388235471854, 'NDCG@12': 0.02458742202128494, 'Coverage': 0.16747834985124405}
Cold: {'Cold MAP@12': 0.0, 'Cold Recall@12': 0.0, 'Cold Recall@100': 0.0011285340936089332}


In [64]:
metadata_comparison = pl.DataFrame([
    {
        "model": "SASRec",
        "MAP@12": 0.014935,
        "Cold Recall@100": 0.0
    },
    {
        "model": "Metadata-SASRec v1",
        "MAP@12": 0.016332,
        "Cold Recall@100": 0.0
    },
    {
        "model": "Metadata-SASRec v2",
        "MAP@12": best_v2_checkpoint["overall_metrics"]["MAP@12"],
        "Cold Recall@100": best_v2_checkpoint["cold_metrics"]["Cold Recall@100"]
    }
]).sort("MAP@12", descending=True)

metadata_comparison

model,MAP@12,Cold Recall@100
str,f64,f64
"""Metadata-SASRec v1""",0.016332,0.0
"""Metadata-SASRec v2""",0.015156,0.001129
"""SASRec""",0.014935,0.0
